### 1) Collect Data

* 1) Google-BigQuery

* 2) hugging-face

#### Load Large 183 GBs dataset

* 1) memory mapping: pointer in RAM to hard-disk (zero copy & zero-overhead)

* 2) Streaming: (uncompress & read on-the-fly) no mermoy nor disk-space needed.

### 2) Build Tokenizer

#### 3) Train a model from scratch

* a) Initialize Model

* b) Implement Dataloader

* c) Custom training loop

* Pretraining Objective (Encoder / Decoder / Seq2Seq):
    - for Encoder: we will make Masked Language Model used to get general representation of the text  as pretrained_encoder like BERT
    - for Decoder: we will make Casual Language Model used to generate future tokens as pretrained_decoder like GPT
    - for Seq2Seq: we will use comments <-> code as pretrained_transformer like BART/T5/M2M/PEGASUS

In [ ]:
def model_size(model):
    return sum(t.numel() for t in model.parameters())

In [4]:
#### * a) Initialize Model

# big GPT
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

model_ckpt = "codeparrot"
tokenizer = AutoTokenizer.from_pretrained(f"ahmed-ayman101/{model_ckpt}")
config = AutoConfig.from_pretrained("gpt2-xl", vocab_size=len(tokenizer))
model = AutoModelForCausalLM.from_config(config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/471 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

In [5]:
print(f'GPT-2 (xl) size: {model_size(model)/1000**2:.1f}M parameters') # 1.5B parameter

GPT-2 (xl) size: 1529.6M parameters


In [7]:
# push big-model to hub
# model.save_pretrained("models/" + model_ckpt, push_to_hub=True)

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  553kB / 1.15GB            

  ...0001-of-00002.safetensors:   0%|          |  554kB / 4.97GB            

In [6]:
# small/standard GPT
tokenizer = AutoTokenizer.from_pretrained(f"ahmed-ayman101/{model_ckpt}")
config_small = AutoConfig.from_pretrained("gpt2", vocab_size=len(tokenizer))
model_small = AutoModelForCausalLM.from_config(config_small)
print(f'GPT-2 size: {model_size(model_small)/1000**2:.1f}M parameters') # 111M parameter

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

GPT-2 size: 111.0M parameters


In [14]:
# push small-model to hub
# model_small.save_pretrained("models/" + model_ckpt + "-small", push_to_hub=True)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t-small/model.safetensors:   0%|          |  556kB /  444MB            

In [15]:
# I already pushed so let's read it
tokenizer = AutoTokenizer.from_pretrained(f"ahmed-ayman101/{model_ckpt}")
model = AutoModelForCausalLM.from_pretrained(f"ahmed-ayman101/{model_ckpt}")
model_small = AutoModelForCausalLM.from_pretrained(f"ahmed-ayman101/{model_ckpt}"+"-small")

config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/874 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/444M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [8]:
#### * b) Implement Dataloader
from tqdm.auto import tqdm
from datasets import load_dataset

examples, total_characters, total_tokens = 500, 0, 0

dataset = load_dataset('transformersbook/codeparrot-train', split='train', streaming=True)

for _, example in tqdm(zip(range(examples), iter(dataset)), total=examples):
    total_characters += len(example['content'])
    total_tokens += len(tokenizer(example['content']).tokens())
    characters_per_token = total_characters / total_tokens
print(characters_per_token)

README.md:   0%|          | 0.00/583 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2599 > 1024). Running this sequence through the model will result in indexing errors


3.6231516195736053


In [9]:
tokenizer(['Ahmed is good', 'Ahmed is bad'], truncation=False)

{'input_ids': [[33, 72, 3525, 365, 6848], [33, 72, 3525, 365, 4896]], 'attention_mask': [[1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]}

In [10]:
import torch
from torch.utils.data import IterableDataset

class ConstantLengthDataset(IterableDataset):
    def __init__(self, tokenizer, dataset, seq_length=1024, num_of_sequences=1024, chars_per_token=3.6):
        self.tokenizer = tokenizer
        self.concat_token_id = tokenizer.eos_token_id
        self.dataset = dataset
        self.seq_length = seq_length
        self.input_characters = num_of_sequences * seq_length * chars_per_token

    def __iter__(self):
        iterator = iter(self.dataset)
        more_examples = True
        while more_examples:
            # fill buffer
            buffer, buffer_len = [], 0
            while True:
                if buffer_len >= self.input_characters:
                    m=f"Buffer full: {buffer_len}>={self.input_characters:.0f}"
                    print(m)
                    break
                try:
                    m=f"Fill buffer: {buffer_len}<{self.input_characters:.0f}"
                    print(m)
                    buffer.append(next(iterator)["content"])
                    buffer_len += len(buffer[-1])

                except StopIteration:
                    iterator = iter(self.dataset)

            # loop over tokenized buffer_of_text
            all_token_ids = []
            tokenized_inputs = self.tokenizer(buffer, truncation=False)
            for tokenized_input in tokenized_inputs["input_ids"]:
                all_token_ids.extend(tokenized_input + [self.concat_token_id])

            # chunk long sequnces into fixed seq_length input
            for i in range(0, len(all_token_ids), self.seq_length):
                input_ids = all_token_ids[i : i + self.seq_length]
                if len(input_ids) == self.seq_length:
                    yield torch.tensor(input_ids)

In [12]:
# so this class specify num_of_sequnces and it will automatically calculate number_of_chars and chunk using sequnce_length
shuffled_dataset = dataset.shuffle(buffer_size=100)
constant_length_dataset = ConstantLengthDataset(tokenizer, shuffled_dataset, num_of_sequences=10)
dataset_iterator = iter(constant_length_dataset)

# test dataloader
lengths = [len(b) for _, b in zip(range(5), dataset_iterator)]
print(f"Lengths of the sequences: {lengths}")

Fill buffer: 0<36864
Buffer full: 197802>=36864
Lengths of the sequences: [1024, 1024, 1024, 1024, 1024]


In [13]:
# * c) Custom training loop (Accelerate for distrubted training & mixed-precision)
# + means for Accelerate
# - means without Accelerate

"""
# just to understand - don't run it
import torch
import torch.nn.functional as F
from datasets import load_dataset
from accelerate import Accelerator # +

device = 'cpu' # -
accelerator = Accelerator() # +
model = torch.nn.Transformer().to(device) # -
model = torch.nn.Transformer() # +
optimizer = torch.optim.Adam(model.parameters())

dataset = load_dataset('my_dataset')
data = torch.utils.data.DataLoader(dataset, shuffle=True)
model, optimizer, data = accelerator.prepare(model, optimizer, data) # +
model.train()
for epoch in range(10):
    for source, targets in data:
        source = source.to(device) # -
        targets = targets.to(device) # -
        optimizer.zero_grad()
        output = model(source)
        loss = F.cross_entropy(output, targets)
        loss.backward() # -
        accelerator.backward(loss) # +
        optimizer.step()
"""
print()

In [14]:
# set hyperparameters
from argparse import Namespace

# Commented parameters correspond to the small model
config = {"train_batch_size": 2, # 12
    "valid_batch_size": 2, # 12
    "weight_decay": 0.1,
    "shuffle_buffer": 1000,
    "learning_rate": 2e-4, # 5e-4
    "lr_scheduler_type": "cosine",
    "num_warmup_steps": 750, # 2000
    "gradient_accumulation_steps": 16, # 1
    "max_train_steps": 50000, # 150000
    "max_eval_steps": -1,
    "seq_length": 1024,
    "seed": 1,
    "save_checkpoint_steps": 50000} # 15000
args = Namespace(**config)

In [16]:
# set logging
from torch.utils.tensorboard import SummaryWriter
import logging
import wandb
def setup_logging(project_name):
    logger = logging.getLogger(__name__)
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
        handlers=[
            logging.FileHandler(f"log/debug_{accelerator.process_index}.log"),
            logging.StreamHandler()
    ])

    if accelerator.is_main_process: # We only want to set up logging once
        wandb.init(project=project_name, config=args)
        run_name = wandb.run.name
        tb_writer = SummaryWriter()
        tb_writer.add_hparams(vars(args), {'0': 0})
        logger.setLevel(logging.INFO)
        datasets.utils.logging.set_verbosity_debug()
        transformers.utils.logging.set_verbosity_info()

    else:
        tb_writer = None
        run_name = ''
        logger.setLevel(logging.ERROR)
        datasets.utils.logging.set_verbosity_error()
        transformers.utils.logging.set_verbosity_error()
    return logger, tb_writer, run_name

In [17]:
def log_metrics(step, metrics):
    logger.info(f"Step {step}: {metrics}")
    if accelerator.is_main_process:
        wandb.log(metrics)
        [tb_writer.add_scalar(k, v, step) for k, v in metrics.items()]

In [18]:
from torch.utils.data.dataloader import DataLoader

def create_dataloaders(dataset_name):
    # read train and valid data
    train_data = load_dataset(dataset_name+'-train', split="train", streaming=True)
    train_data = train_data.shuffle(buffer_size=args.shuffle_buffer, seed=args.seed)
    valid_data = load_dataset(dataset_name+'-valid', split="validation", streaming=True)

    # constant sequnce length
    train_dataset = ConstantLengthDataset(tokenizer, train_data, seq_length=args.seq_length)
    valid_dataset = ConstantLengthDataset(tokenizer, valid_data, seq_length=args.seq_length)

    # batch_dataset using torch DataLoader
    train_dataloader = DataLoader(train_dataset, batch_size=args.train_batch_size)
    eval_dataloader = DataLoader(valid_dataset, batch_size=args.valid_batch_size)
    return train_dataloader, eval_dataloader

In [19]:
# prepare weight_decay for all layers except bias and LayerNorm
def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [{'params': params_with_wd, 'weight_decay': args.weight_decay}, {'params': params_without_wd, 'weight_decay': 0.0}]

In [20]:
# Evaluate function contains (loss & perplexity)
def evaluate():
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(batch, labels=batch)
        loss = outputs.loss.repeat(args.valid_batch_size)
        losses.append(accelerator.gather(loss))
        if args.max_eval_steps > 0 and step >= args.max_eval_steps: break

    loss = torch.mean(torch.cat(losses))
    try:
       perplexity = torch.exp(loss)
    except OverflowError:
        perplexity = torch.tensor(float("inf"))
    return loss.item(), perplexity.item()

In [24]:
# Custom Training Loop

# from transformers import set_seed
# project_name = 'ahmed-ayman101/codeparrot'
# dataset_name = '../codeparrot'
# in codeparrot_training.py

In [25]:
# the model does not fit on a single GPU you might need more sophisticated
# https://huggingface.co/docs/transformers/v4.13.0/parallelism

In [ ]:
# run commands
# $ git clone https://huggingface.co/transformersbook/codeparrot
# $ cd codeparrot
# $ pip install -r requirements.txt
# $ wandb login
# $ accelerate config
# $ accelerate launch codeparrot_training.py

In [26]:
# after training
from transformers import pipeline, set_seed

model_ckpt = 'transformersbook/codeparrot-small'
generation = pipeline('text-generation', model=model_ckpt, device=0)

config.json:   0%|          | 0.00/865 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/457M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Device set to use cpu


In [31]:
import re
from transformers import set_seed

def first_block(string):
    return re.split('\nclass|\ndef|\n#|\n@|\nprint|\nif', string)[0].rstrip()

def complete_code(pipe, prompt, max_length=64, num_completions=4, seed=1):
    set_seed(seed)
    gen_kwargs = {"temperature":0.4, "top_p":0.95, "top_k":0, "num_beams":1, "do_sample":True,}
    code_gens = generation(prompt, num_return_sequences=num_completions, truncation=True, max_new_tokens=max_length, **gen_kwargs)
    code_strings = []
    for code_gen in code_gens:
        generated_code = first_block(code_gen['generated_text'][len(prompt):])
        code_strings.append(generated_code)
    print(('\n'+'='*80 + '\n').join(code_strings))

In [32]:
prompt = '''def area_of_rectangle(a: float, b: float):
    """Return the area of the rectangle."""'''
complete_code(generation, prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



    return a * b / (a + b)

    return a * b

    return (a + b) / 2.0

    return a / b


In [33]:
# more diffcult task
prompt = '''def get_urls_from_html(html):
    """Get all embedded URLs in a HTML string."""'''
complete_code(generation, prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



    return [url for url in re.findall(r'<a href="([^"']+)[^>]*?>(.*?)</a>', html)
            if not url.startswith('http')]

    return re.findall(r'<a href="(.*?)"', html, re.DOTALL)

    if not isinstance(html, basestring):
        html = str(html)

    return [url for url in re.findall(r'<a href="\1" class="url">', html) if url]

    return [url for url in re.findall(r'<a href="(/[^/]+/[^/]+/[^/]+/[^/?#]+?)"', html) if url]


In [39]:
import requests
import re

def get_urls_from_html(html):
    return re.findall(r'<a href="(.*?)"', html, re.DOTALL)


print(" | ".join(get_urls_from_html(requests.get('https://hf.co/').text)))

/spaces | /models | /models | /spaces/Wan-AI/Wan2.2-Animate | /spaces/microsoft/TRELLIS.2 | /spaces/Qwen/Qwen-Image-Layered | /spaces/selfit-camera/Omni-Image-Editor | /spaces/mrfakename/Z-Image-Turbo | /spaces | /datasets | /enterprise | /enterprise | /enterprise | /enterprise | /enterprise | /enterprise | /docs/transformers | /docs/diffusers | /docs/safetensors | /docs/huggingface_hub | /docs/tokenizers | /docs/trl | /docs/transformers.js | /docs/smolagents | /docs/peft | /docs/datasets | /docs/text-generation-inference | /docs/accelerate


In [40]:
# Large Model
model_ckpt = 'transformersbook/codeparrot'
generation = pipeline('text-generation', model=model_ckpt, device=0)

config.json:   0%|          | 0.00/959 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



    return np.mean(a)

    return np.mean(a)

    return np.mean(a)

    return np.mean(a)


In [41]:
# convert python code to numpy
prompt = '''# a function in native python:
def mean(a):
    return sum(a)/len(a)
# the same function using numpy:
import numpy as np
def mean(a):'''

complete_code(generation, prompt, max_length=64)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



    return np.mean(a)

    return np.mean(a)

    return np.mean(a)

    return np.mean(a)


In [42]:
# fit sickit-learn model
prompt = '''X = np.random.randn(100, 100)
y = np.random.randint(0, 1, 100)
# fit random forest classifier with 20 estimators'''

complete_code(generation, prompt, max_length=96)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



clf = RandomForestClassifier(n_estimators=20)
clf.fit(X, y)

clf = RandomForestClassifier(n_estimators=20, random_state=1)
clf.fit(X, y)

clf = RandomForestClassifier(n_estimators=20, n_jobs=20, random_state=1)
clf.fit(X, y)

clf = RandomForestClassifier(n_estimators=20, random_state=1)
clf.fit(X, y)
